<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/MultiModal/vlm_obj_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -Uq transformers datasets trl supervision albumentations

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset
refcoco_dataset = load_dataset("jxu124/refcoco",split='train[:5%]')

In [ ]:
refcoco_dataset[13]['raw_image_info']


In [ ]:
import json
import requests
from PIL import Image
from io import BytesIO

def add_image(example):
    try:
        raw_info = json.loads(example['raw_image_info'])
        url = raw_info.get('flickr_url', None)
        if url:
            response = requests.get(url, timeout=10)
            image = Image.open(BytesIO(response.content)).convert("RGB")
            example['image'] = image
        else:
            example['image'] = None
    except Exception as e:
        print(f"Error loading image: {e}")
        example['image'] = None
    return example

refcoco_dataset_with_images = refcoco_dataset.map(add_image, desc="Adding image from flickr", num_proc=16)


In [ ]:
filtered_dataset = refcoco_dataset_with_images.filter(
    lambda example: example['image'] is not None,
    desc="Removing failed image downloads"
)

In [ ]:
filtered_dataset = filtered_dataset.remove_columns(['sent_ids', 'file_name', 'ann_id', 'ref_id', 'image_id', 'split', 'sentences', 'category_id', 'raw_anns', 'raw_image_info', 'raw_sentences', 'image_path', 'global_image_id', 'anns_id'])


In [ ]:
def separate_captions_into_unique_samples(batch):
    new_images = []
    new_bboxes = []
    new_captions = []

    for image, bbox, captions in zip(batch["image"], batch["bbox"], batch["captions"]):
        for caption in captions:
            new_images.append(image)
            new_bboxes.append(bbox)
            new_captions.append(caption)

    return {
        "image": new_images,
        "bbox": new_bboxes,
        "caption": new_captions,
    }

filtered_dataset = filtered_dataset.map(
    separate_captions_into_unique_samples,
    batched=True,
    batch_size=100,
    num_proc=4,
    remove_columns=filtered_dataset.column_names
)

In [ ]:
filtered_dataset[20]['caption']


In [ ]:
filtered_dataset[20]['bbox']

In [ ]:
filtered_dataset[20]['image']

In [ ]:
labels = [(filtered_dataset[20]['caption'], filtered_dataset[20]['bbox'])]

In [ ]:
import supervision as sv
import numpy as np

In [ ]:
def get_annotated_image(image, parsed_labels):
    if not parsed_labels:
        return image

    xyxys = []
    labels = []

    for label, bbox in parsed_labels:
        xyxys.append(bbox)
        labels.append(label)

    detections = sv.Detections(xyxy=np.array(xyxys))

    bounding_box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.INDEX)
    label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.INDEX)

    annotated_image = bounding_box_annotator.annotate(
        scene=image, detections=detections
    )
    annotated_image = label_annotator.annotate(
        scene=annotated_image, detections=detections, labels=labels
    )

    return annotated_image


In [ ]:
annotated_image = get_annotated_image(filtered_dataset[20]['image'], labels)
annotated_image

In [ ]:
split_dataset = filtered_dataset.train_test_split(test_size=0.2, seed=42, shuffle=False)
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']
train_dataset, val_dataset


In [ ]:
from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
)
import torch

model_id = "google/paligemma2-3b-pt-448"

model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto").eval()
processor = PaliGemmaProcessor.from_pretrained(model_id, use_fast=True)
